# pysmatch: reproducible matching workflow

This notebook demonstrates batched scoring, logit calipers, matching diagnostics, and optional matching strategies. Install with `pip install "pysmatch[all]"`.

In [ ]:
from pysmatch import Matcher, load_demo

data = load_demo()
test = data[data.loan_status == "Default"].copy()
control = data[data.loan_status == "Fully Paid"].copy()

matcher = Matcher(
    test=test,
    control=control,
    yvar="is_default",
    exclude=["loan_status"],
    random_state=42,
)

In [ ]:
matcher.fit_scores(
    balance=True,
    nmodels=2,
    model_type="linear",
    n_jobs=2,
    random_state=42,
)
matcher.predict_scores(batch_size=128)
print(matcher.score_report())
print(matcher.calibration_report())

In [ ]:
matched = matcher.match(
    method="min",
    distance="logit",
    threshold=0.2,
    nmatches=1,
    replacement=False,
    chunk_size=128,
)
print("matched rows:", len(matched))
print(matcher.overlap_report())
print(matcher.positivity_report())
print(matcher.balance_summary())
print(matcher.effective_sample_size())

In [ ]:
matcher.plot_love(threshold=0.1)
matcher.plot_overlap(scope="after")
matcher.plot_calibration()

## Alternative strategies

```python
# matcher.match(threshold=0.05, exact=["grade"], chunk_size=128)
# matcher.match(method="optimal", threshold=0.05)
# effect = matcher.estimate_effect("outcome", estimand="ATT", bootstrap=1000)
```